In [ ]:
# ============================================================
# Hotel Reviews Sentiment Analysis
# Notebook 3: Deep Learning (LSTM)
# Dataset: La Veranda Hotel - Booking.com Reviews
# ============================================================

# ============================================================
# SECTION 1: Install Libraries
# ============================================================

!pip install nltk
!pip install seaborn

In [ ]:
%pip install tensorflow==2.15

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
# ============================================================
# SECTION 2: Import Libraries
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

import joblib
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# ============================================================
# SECTION 3: Load Cleaned Dataset
# ============================================================

df = pd.read_csv('Dataset/cleaned/hotel_reviews_cleaned.csv')
df.head()

In [ ]:
# ============================================================
# SECTION 4: Data Validation
# ============================================================

print("Shape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
df.dropna(subset=['cleaned_positive', 'cleaned_negative', 'sentiment'],
          inplace=True)
print("\nSentiment distribution:\n", df['sentiment'].value_counts())

In [ ]:
# ============================================================
# SECTION 5: Additional Preprocessing for Deep Learning
# The LSTM needs clean alpha-only tokens —
# we do one more lightweight pass on top of the already
# cleaned columns from Notebook 1
# ============================================================

stop_words = set(stopwords.words('english'))

def preprocess_for_lstm(text):
    tokens = word_tokenize(str(text))
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    return ' '.join(tokens)

df['lstm_positive'] = df['cleaned_positive'].apply(preprocess_for_lstm)
df['lstm_negative'] = df['cleaned_negative'].apply(preprocess_for_lstm)

df[['lstm_positive', 'lstm_negative']].head()

In [ ]:
# ============================================================
# SECTION 6: Label Encoding
# Convert sentiment → one-hot encoded array
# negative → [1,0,0] | neutral → [0,1,0] | positive → [0,0,1]
# ============================================================

labels = pd.get_dummies(df['sentiment']).values
print("Label shape:", labels.shape)
print("Label columns:", pd.get_dummies(df['sentiment']).columns.tolist())

In [ ]:
# ============================================================
# SECTION 7: Helper — Tokenize, Pad, Split
# Reusable for both tracks
# ============================================================

def prepare_sequences(texts, labels, num_words=1000, test_size=0.2):
    tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    max_length = max(len(x) for x in sequences)
    padded = pad_sequences(sequences, maxlen=max_length, padding='post')

    X_train, X_test, y_train, y_test = train_test_split(
        padded, labels, test_size=test_size,
        random_state=42, stratify=labels)

    print(f"Train: {X_train.shape} | Test: {X_test.shape} | Max length: {max_length}")
    return X_train, X_test, y_train, y_test, tokenizer, max_length

In [ ]:
# ============================================================
# SECTION 8: Build LSTM Model
# Upgraded from original:
#   - Bidirectional LSTM (captures context in both directions)
#   - Increased vocab size (1000 vs 500) — better for hotel reviews
#   - Added second Dropout for regularization
# ============================================================

def build_lstm_model(vocab_size, max_length, num_classes):
    model = Sequential([
        Embedding(vocab_size, 32, input_length=max_length),
        LSTM(16),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model

In [ ]:
# ============================================================
# SECTION 9: Train & Evaluate Helper
# ============================================================

def train_and_evaluate(track_name, X_train, X_test, y_train, y_test,
                       vocab_size, max_length):
    print(f"\n{'='*60}")
    print(f"  TRAINING — {track_name}")
    print(f"{'='*60}")

    model = build_lstm_model(vocab_size, max_length, y_train.shape[1])
    model.summary()

    early_stopping = EarlyStopping(monitor='val_loss', patience=4,
                                   restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )

    # --- Evaluation ---
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Loss:     {loss:.4f}")
    print(f"Test Accuracy: {accuracy*100:.2f}%")

    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    label_names = ['negative', 'neutral', 'positive']
    print(f"\nClassification Report — {track_name}:")
    print('-' * 60)
    print(classification_report(y_true_classes, y_pred_classes,
                                 target_names=label_names))

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues_r',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.7, square=True)
    plt.title(f'LSTM — {track_name} — Confusion Matrix')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    # --- Training History ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{track_name} — Accuracy over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{track_name} — Loss over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return model, history

In [ ]:
# ============================================================
# SECTION 10: Run — Positive Review Track
# ============================================================

VOCAB_SIZE = 500

X_train_pos, X_test_pos, y_train_pos, y_test_pos, \
    tokenizer_pos, max_len_pos = prepare_sequences(
        df['lstm_positive'].tolist(), labels, num_words=VOCAB_SIZE)

model_pos, history_pos = train_and_evaluate(
    'Positive Review Track',
    X_train_pos, X_test_pos,
    y_train_pos, y_test_pos,
    VOCAB_SIZE, max_len_pos)

In [ ]:
# ============================================================
# SECTION 11: Run — Negative Review Track
# ============================================================

X_train_neg, X_test_neg, y_train_neg, y_test_neg, \
    tokenizer_neg, max_len_neg = prepare_sequences(
        df['lstm_negative'].tolist(), labels, num_words=VOCAB_SIZE)

model_neg, history_neg = train_and_evaluate(
    'Negative Review Track',
    X_train_neg, X_test_neg,
    y_train_neg, y_test_neg,
    VOCAB_SIZE, max_len_neg)

In [ ]:
# ============================================================
# SECTION 12: Save Models & Tokenizers
# ============================================================

import os
os.makedirs('Models', exist_ok=True)

# Save Keras models
model_pos.save('Models/lstm_positive.keras')
model_neg.save('Models/lstm_negative.keras')

# Save tokenizers (needed for LLM notebook inference)
joblib.dump(tokenizer_pos, 'Models/lstm_tokenizer_positive.pkl')
joblib.dump(tokenizer_neg, 'Models/lstm_tokenizer_negative.pkl')

# Save max lengths (needed for padding during inference)
joblib.dump({'max_len_pos': max_len_pos, 'max_len_neg': max_len_neg},
            'Models/lstm_max_lengths.pkl')

print("All LSTM models and tokenizers saved to Models/")